In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

data = pd.read_csv("/content/mushrooms.csv")
print("Shape:", data.shape)
data.head()

Shape: (8124, 23)


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [ ]:
print("Parth Madrewar\nReg No. 241080036\nBranch: IT")

Parth Madrewar
Reg No. 241080036
Branch: IT


In [ ]:
# ID3 works fine on categorical data directly, but encoding keeps things
# consistent and lets us reuse the same dataframe for sklearn's tree later.
encoders = {}
encoded_data = data.copy()

for col in encoded_data.columns:
    enc = LabelEncoder()
    encoded_data[col] = enc.fit_transform(encoded_data[col])
    encoders[col] = enc

encoded_data.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,1,5,2,4,1,6,1,0,1,4,...,2,7,7,0,2,1,4,2,3,5
1,0,5,2,9,1,0,1,0,0,4,...,2,7,7,0,2,1,4,3,2,1
2,0,0,2,8,1,3,1,0,0,5,...,2,7,7,0,2,1,4,3,2,3
3,1,5,3,8,1,6,1,0,1,5,...,2,7,7,0,2,1,4,2,3,5
4,0,5,2,3,0,5,1,1,0,4,...,2,7,7,0,2,1,0,3,0,1


In [ ]:
# "class" is the target: edible (e) vs poisonous (p), encoded as 0/1.
# Show the mapping so it's explicit what 0 and 1 mean.
class_mapping = dict(zip(encoders['class'].classes_, encoders['class'].transform(encoders['class'].classes_)))
print("Class mapping:", class_mapping)

class_counts = data['class'].value_counts()
print("\nClass distribution:\n", class_counts)

# Class as a Boolean expression over the raw attributes (example form):
# class = poisonous  IF  odor = foul  OR  odor = pungent  OR  (odor = none AND spore-print-color = green) ...
# We don't hardcode this — it's exactly what the ID3 tree we build below discovers automatically,
# and we'll print the learned rules from the tree in Cell 7.

Class mapping: {'e': np.int64(0), 'p': np.int64(1)}

Class distribution:
 class
e    4208
p    3916
Name: count, dtype: int64


In [ ]:
class TreeNode:
    def __init__(self, feature=None, branches=None, prediction=None):
        self.feature = feature        # feature this node splits on (None if leaf)
        self.branches = branches if branches is not None else {}  # {feature_value: TreeNode}
        self.prediction = prediction  # class label if leaf


def compute_entropy(labels):
    """Entropy H(S) = -sum(p_i * log2(p_i)) for each class i."""
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))


def compute_info_gain(feature_col, labels):
    """Information Gain = H(S) - sum over feature values of (|Sv|/|S|) * H(Sv)."""
    total_entropy = compute_entropy(labels)
    values, counts = np.unique(feature_col, return_counts=True)

    weighted_entropy = 0
    for v, c in zip(values, counts):
        subset_labels = labels[feature_col == v]
        weighted_entropy += (c / len(labels)) * compute_entropy(subset_labels)

    return total_entropy - weighted_entropy

In [ ]:
X_full = encoded_data.drop(columns=["class"])
y_full = encoded_data["class"]

print(f"Entropy of full dataset (target 'class'): {compute_entropy(y_full):.4f}\n")

gains = {}
for col in X_full.columns:
    gains[col] = compute_info_gain(X_full[col], y_full)

gain_table = pd.DataFrame(sorted(gains.items(), key=lambda x: x[1], reverse=True),
                           columns=["Feature", "Information Gain"])
print("Information Gain per feature (root node):")
gain_table

Entropy of full dataset (target 'class'): 0.9991

Information Gain per feature (root node):


,Feature,Information Gain
0,odor,0.906075
1,spore-print-color,0.480705
2,gill-color,0.416978
3,ring-type,0.318022
4,stalk-surface-above-ring,0.284726
5,stalk-surface-below-ring,0.271894
6,stalk-color-above-ring,0.253845
7,stalk-color-below-ring,0.241416
8,gill-size,0.230154
9,population,0.201958


In [ ]:
best_root_feature = gain_table.iloc[0]["Feature"]
print(f"Feature selected at root: '{best_root_feature}' (highest information gain)")

Feature selected at root: 'odor' (highest information gain)


In [ ]:
def build_id3(X, y):
    # Case 1: all labels identical -> pure leaf
    if len(y.unique()) == 1:
        return TreeNode(prediction=y.iloc[0])

    # Case 2: no features left -> leaf with majority class
    if X.shape[1] == 0:
        return TreeNode(prediction=y.mode()[0])

    # Choose the feature with highest information gain
    best_feature, best_gain = None, -1
    for col in X.columns:
        g = compute_info_gain(X[col], y)
        if g > best_gain:
            best_gain, best_feature = g, col

    node = TreeNode(feature=best_feature)

    for val in np.unique(X[best_feature]):
        mask = X[best_feature] == val
        X_subset = X[mask].drop(columns=[best_feature])
        y_subset = y[mask]

        if len(y_subset) == 0:
            node.branches[val] = TreeNode(prediction=y.mode()[0])
        else:
            node.branches[val] = build_id3(X_subset, y_subset)

    return node


def predict_one(node, row):
    if node.prediction is not None:
        return node.prediction
    value = row[node.feature]
    if value not in node.branches:
        # unseen value at this branch during test time -> fall back to majority guess of 0
        return 0
    return predict_one(node.branches[value], row)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

id3_root = build_id3(X_train, y_train)

predictions = [predict_one(id3_root, X_test.iloc[i]) for i in range(len(X_test))]

custom_accuracy = accuracy_score(y_test, predictions)
print(f"Custom ID3 Accuracy: {custom_accuracy * 100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))

Custom ID3 Accuracy: 100.00%

Confusion Matrix:
 [[843   0]
 [  0 782]]


In [ ]:
sk_tree = DecisionTreeClassifier(criterion="entropy", random_state=42)
sk_tree.fit(X_train, y_train)
sk_predictions = sk_tree.predict(X_test)

sk_accuracy = accuracy_score(y_test, sk_predictions)
print(f"Sklearn Decision Tree Accuracy: {sk_accuracy * 100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, sk_predictions))

print("\n--- Comparison ---")
print(f"Custom ID3 Accuracy   : {custom_accuracy * 100:.2f}%")
print(f"Sklearn Tree Accuracy : {sk_accuracy * 100:.2f}%")

Sklearn Decision Tree Accuracy: 100.00%

Confusion Matrix:
 [[843   0]
 [  0 782]]

--- Comparison ---
Custom ID3 Accuracy   : 100.00%
Sklearn Tree Accuracy : 100.00%
